In [ ]:
#Lets have matplotlib "inline"
%matplotlib inline

# Add line profiler
%load_ext line_profiler

#Import packages we need
from matplotlib import animation, rc
from matplotlib import pyplot as plt

import importlib

try:
    from StringIO import StringIO
except ImportError:
    from io import StringIO
    
#Set large figure sizes
#Note, this prevents nice figures for articles...
rc('figure', figsize=(16.0, 12.0))
rc('animation', html='html5')

from GPUSimulators.common import Timer
from GPUSimulators.helpers import initial_conditions

In [ ]:
%setup_logging --out=test_schemes.log TestSchemes

In [3]:
%cuda_context_handler my_context

Registering my_context in user workspace
PyCUDA version 2024.1
CUDA version (11, 8, 0)
Driver version 12080
Using device 0/1 'NVIDIA GeForce GTX 970' (0000:07:00.0) GPU


Created context handle <98151551869168>
Using CUDA cache dir /home/smyalygames/PycharmProjects/FiniteVolumeGPU/GPUSimulators/cuda_cache
Autotuning enabled. It may take several minutes to run the code the first time: have patience


In [4]:
nx = 256
ny = 128
g = 9.81

h0, hu0, hv0, dx, dy = InitialConditions.bump(nx, ny, 100, 100, 15)

# print(my_context.autotuner.get)

arguments = {
    'context': my_context,
    'h0': h0, 'hu0': hu0, 'hv0': hv0,
    'nx': nx, 'ny': ny,
    'dx': dx, 'dy': dy, 
    'g': g,
    'compile_opts': ['-Wno-deprecated-gpu-targets', '-arch=sm_52']
}

t_end = 20

In [5]:
def plot(sim):
    h = sim.u0[0].download(sim.stream)
    
    plt.figure()
    plt.title(str(sim) + ", t=" + str(sim.sim_time()) + ", nt=" + str(sim.sim_steps()))
    extent = [0, sim.dx*sim.nx, 0, sim.dy*sim.ny]
    plt.imshow(h, vmin=0.49, vmax=0.52, extent=extent)
    plt.colorbar()

In [ ]:
from GPUSimulators.model import LxF

with Timer("construct") as t:
    sim = LxF(**arguments)

with Timer("step") as t:
    t = sim.simulate(t_end)
    
with Timer("download") as t:
    h1, hu1, hv1 = sim.download()

plot(sim)

In [ ]:
from GPUSimulators.model import Force

with Timer("construct") as t:
    sim = Force(**arguments)

with Timer("step") as t:
    t = sim.simulate(t_end)
    
with Timer("download") as t:
    h1, hu1, hv1 = sim.download()

plot(sim)

In [ ]:
from GPUSimulators.model import HLL

with Timer("construct") as t:
    sim = HLL(**arguments)

with Timer("step") as t:
    t = sim.simulate(t_end)
    
with Timer("download") as t:
    h1, hu1, hv1 = sim.download()

plot(sim)

In [ ]:
from GPUSimulators.model import HLL2

with Timer("construct") as t:
    sim = HLL2(**arguments)

with Timer("step") as t:
    t = sim.simulate(t_end)
    
with Timer("download") as t:
    h1, hu1, hv1 = sim.download()

plot(sim)

In [ ]:
from GPUSimulators.model import KP07

with Timer("construct") as t:
    sim = KP07(**arguments)

with Timer("step") as t:
    t = sim.simulate(t_end)
    
with Timer("download") as t:
    h1, hu1, hv1 = sim.download()

plot(sim)

In [ ]:
from GPUSimulators.model import KP07Dimsplit

with Timer("construct") as t:
    sim = KP07Dimsplit(**arguments)

with Timer("step") as t:
    t = sim.simulate(t_end)
    
with Timer("download") as t:
    h1, hu1, hv1 = sim.download()

plot(sim)

In [ ]:
from GPUSimulators.model import WAF

with Timer("construct") as t:
    sim = WAF(**arguments)

with Timer("step") as t:
    t = sim.simulate(t_end)
    
with Timer("download") as t:
    h1, hu1, hv1 = sim.download()

plot(sim)